# Training kandidat tactile paving v3 (Colab)

Notebook bersih untuk melatih kandidat **2:1** yang belum selesai. Kandidat 4:1 tidak diulang.

Sebelum memilih **Jalankan semua**:
1. Pilih runtime **GPU T4**.
2. Unggah `guidetwsi-rbar-v1.zip` melalui panel File. Lokasi `/content` atau `/` sama-sama didukung.

Output tetap kandidat offline dan belum boleh dipakai sebagai jaminan keselamatan sebelum lolos evaluasi protected ground truth.

In [ ]:
!pip install -q "ultralytics==8.4.138" opencv-python-headless
!command -v git-lfs >/dev/null || (apt-get update -qq && apt-get install -y -qq git-lfs)
!git lfs install

In [ ]:
from pathlib import Path, PurePosixPath
import shutil
import subprocess
import sys

REPO = Path('/content/Yoloooo')
BRANCH = 'codex/model-first-mobile-ready'

if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch',
        'https://github.com/Riqqi15/Yoloooo.git', str(REPO),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)

subprocess.run(['git', '-C', str(REPO), 'lfs', 'pull'], check=True)

def run_script(name, *arguments):
    subprocess.run(
        [sys.executable, str(REPO / 'scripts' / name), *map(str, arguments)],
        cwd=REPO,
        check=True,
    )

In [ ]:
from zipfile import ZipFile

CACHE_PARENT = REPO / 'runs/public-data-cache'
CACHE = CACHE_PARENT / 'guidetwsi-rbar-v1'
EXPECTED_CACHE_FILES = 3960
DATA_ARCHIVES = (
    Path('/content/guidetwsi-rbar-v1.zip'),
    Path('/guidetwsi-rbar-v1.zip'),
)

archive = next((path for path in DATA_ARCHIVES if path.is_file()), None)
cache_files = [path for path in CACHE.rglob('*') if path.is_file()]

if len(cache_files) != EXPECTED_CACHE_FILES:
    if archive is None:
        raise FileNotFoundError(
            'Upload guidetwsi-rbar-v1.zip ke panel File, lalu jalankan cell ini lagi.'
        )
    shutil.rmtree(CACHE, ignore_errors=True)
    CACHE_PARENT.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive) as zip_file:
        for member in zip_file.infolist():
            member_path = PurePosixPath(member.filename)
            if (
                member_path.is_absolute()
                or '..' in member_path.parts
                or (member_path.parts and member_path.parts[0].endswith(':'))
            ):
                raise ValueError(f'Path ZIP tidak aman: {member.filename}')
        zip_file.extractall(CACHE_PARENT)
    cache_files = [path for path in CACHE.rglob('*') if path.is_file()]

if len(cache_files) != EXPECTED_CACHE_FILES:
    raise ValueError(
        f'Isi cache tidak lengkap: {len(cache_files)}/{EXPECTED_CACHE_FILES} file'
    )

print(f'Dataset publik siap: {len(cache_files)} file')

In [ ]:
PUBLIC_MANIFEST = REPO / 'data/training/manifests/guidetwsi-rbar-2k-v1.json'
TRAIN_RATIOS = (2,)

if not PUBLIC_MANIFEST.is_file():
    raise FileNotFoundError(PUBLIC_MANIFEST)

for ratio in TRAIN_RATIOS:
    version = f'tactile-v3-public{ratio}-station1'
    output = REPO / 'artifacts/datasets' / version
    dataset_yaml = output / 'dataset.yaml'
    manifest = REPO / 'data/training/manifests' / f'{version}.json'
    if not dataset_yaml.is_file():
        shutil.rmtree(output, ignore_errors=True)
        run_script(
            'build_tactile_v3_dataset.py',
            '--output-root', output,
            '--manifest-output', manifest,
            '--public-to-station', ratio,
        )
    print(f'Dataset siap: {version}')

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU tidak aktif. Pilih Runtime > Ubah jenis runtime > GPU T4.')

CHECKPOINT = REPO / 'models/guidetwsi/yolo11n_tactile.pt'
if not CHECKPOINT.is_file():
    raise FileNotFoundError(CHECKPOINT)

print('GPU:', torch.cuda.get_device_name(0))

for ratio in TRAIN_RATIOS:
    version = f'tactile-v3-public{ratio}-station1'
    run_name = f'tactile-one-class-v3-public{ratio}-station1'
    run_dir = REPO / 'runs/segment' / run_name
    candidate = REPO / 'artifacts/candidates' / run_name

    if (candidate / 'best.pt').is_file():
        print(f'Kandidat sudah ada, training dilewati: {run_name}')
        continue

    shutil.rmtree(candidate, ignore_errors=True)
    shutil.rmtree(run_dir, ignore_errors=True)
    run_script(
        'train_tactile_v3.py',
        '--dataset', REPO / 'artifacts/datasets' / version / 'dataset.yaml',
        '--checkpoint', CHECKPOINT,
        '--run-name', run_name,
        '--runs-root', REPO / 'runs/segment',
        '--candidate-root', REPO / 'artifacts/candidates',
        '--device', '0',
        '--epochs', '80',
        '--batch', '8',
    )

In [ ]:
from google.colab import files

REQUIRED_OUTPUTS = {'best.pt', 'metrics.json', 'training_config.json', 'MODEL_CARD.md', 'sha256.txt'}

for ratio in TRAIN_RATIOS:
    run_name = f'tactile-one-class-v3-public{ratio}-station1'
    candidate = REPO / 'artifacts/candidates' / run_name
    missing = REQUIRED_OUTPUTS - {path.name for path in candidate.iterdir()}
    if missing:
        raise FileNotFoundError(f'Output kandidat belum lengkap: {sorted(missing)}')
    archive = shutil.make_archive(f'/content/{run_name}', 'zip', candidate)
    print('Model berhasil dibuat:', archive)
    files.download(archive)